1-1. 교통 민원 자동 분류 및 조치안 생성 시스템

In [2]:
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

load_dotenv()

# 1. Pydantic 스키마 정의
class TrafficComplaint(BaseModel):
    location: str = Field(description="민원 발생 위치 또는 도로명/교차로명")
    category: str = Field(description="민원 카테고리 (예: 신호운영, 보행자안전, 도로파손, 교통혼잡 등)")
    urgency_score: int = Field(description="1~5점 사이의 긴급도/위험도 점수")
    action_items: List[str] = Field(description="담당 부서에서 검토해야 할 현장 조치 방안 리스트")

# 2. 파서 및 프롬프트 작성
parser = PydanticOutputParser(pydantic_object=TrafficComplaint)

prompt = PromptTemplate(
    template="다음 교통 민원 내용에서 주요 정보를 구조화하여 추출하세요.\n민원내용: {complaint}\n{format_instructions}",
    input_variables=["complaint"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
chain = prompt | llm | parser

# 3. 실행 예시
complaint_text = "OO사거리 신호등 보행 시간이 너무 짧아서 어르신들이 건너기 위험합니다. 횡단보도 보행신호 시간을 늘려주거나 보조신호등을 설치해주세요."
result = chain.invoke({"complaint": complaint_text})

print("=== [아이디어 1] 교통 민원 분석 결과 ===")
# 기존: print(result.model_dump())

print("=== [아이디어 1] 교통 민원 분석 결과 ===")
print(f"📍 민원 위치: {result.location}")
print(f"📁 분류 항목: {result.category}")
print(f"🚨 긴급도 점수: {result.urgency_score} / 5점")
print("🛠️ 추천 조치 사항:")
for idx, item in enumerate(result.action_items, 1):
    print(f"   {idx}. {item}")

=== [아이디어 1] 교통 민원 분석 결과 ===
=== [아이디어 1] 교통 민원 분석 결과 ===
📍 민원 위치: OO사거리
📁 분류 항목: 보행자안전
🚨 긴급도 점수: 4 / 5점
🛠️ 추천 조치 사항:
   1. 횡단보도 보행신호 시간 연장
   2. 보조신호등 설치


1-2. 도시정비사업 주민 제안서 & 분석 요약기

In [3]:
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

load_dotenv()

# 1. Pydantic 스키마 정의
class UrbanRedevelopmentPlan(BaseModel):
    zone_name: str = Field(description="정비구역 명칭")
    area_sqm: str = Field(description="구역 면적(㎡ 또는 평)")
    target_far: str = Field(description="계획 용적률(%)")
    infrastructure: List[str] = Field(description="기반시설 및 기부채납 항목 리스트")
    key_issues: List[str] = Field(description="주요 민원 또는 개발 이슈 리스트")

# 2. 파서 및 프롬프트 작성
parser = PydanticOutputParser(pydantic_object=UrbanRedevelopmentPlan)

prompt = PromptTemplate(
    template="다음 도시정비사업 고시/제안서 텍스트에서 주요 핵심 정보를 추출하세요.\n텍스트: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
chain = prompt | llm | parser

# 3. 실행 예시
redevelopment_text = "본 사업은 신림1구역 재개발정비사업으로 총 면적 224,773㎡를 대상으로 합니다. 용적률 259.8%를 적용하여 공동주택을 건설할 예정이며, 기부채납 시설로 공공청사와 공용주차장을 확보합니다. 주민들은 세대수 확대 및 단지 내 도로 폭 확장을 지속적으로 요구하고 있습니다."
result = chain.invoke({"text": redevelopment_text})

print("=== [아이디어 2] 도시정비사업 분석 결과 ===")
print(result.model_dump())

=== [아이디어 2] 도시정비사업 분석 결과 ===
{'zone_name': '신림1구역', 'area_sqm': '224,773㎡', 'target_far': '259.8%', 'infrastructure': ['공공청사', '공용주차장'], 'key_issues': ['세대수 확대 요구', '단지 내 도로 폭 확장 요구']}


1-3. 스마트 주차장 및 공공부지 입지 평가 요약기

In [4]:
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

load_dotenv()

# 1. Pydantic 스키마 정의
class SiteEvaluation(BaseModel):
    site_id: str = Field(description="후보지 명칭 또는 위치")
    land_area: str = Field(description="부지 규모 및 면적")
    suitability_score: int = Field(description="1~5점 사이의 입지 적합도 점수")
    pros: List[str] = Field(description="입지 장점 리스트")
    cons: List[str] = Field(description="입지 단점/제약사항 리스트")

# 2. 파서 및 프롬프트 작성
parser = PydanticOutputParser(pydantic_object=SiteEvaluation)

prompt = PromptTemplate(
    template="다음 부지 평가 조사서에서 입지 분석 결과를 추출하세요.\n조사서: {report}\n{format_instructions}",
    input_variables=["report"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
chain = prompt | llm | parser

# 3. 실행 예시
site_report = "A 후보지는 주택가 밀집 지역으로 야간 주차난이 매우 심각한 구역입니다. 부지 면적은 약 1,200㎡이며 진출입로의 가시 확보가 우수합니다. 다만 주변 진입도로 폭이 6m로 협소하여 대형 차량의 진입이 제한적이라는 단점이 있습니다. 전체적인 입지 적합도는 4점으로 평가됩니다."
result = chain.invoke({"report": site_report})

print("=== [아이디어 3] 부지 평가 결과 ===")
print(result.model_dump())

=== [아이디어 3] 부지 평가 결과 ===
{'site_id': 'A 후보지', 'land_area': '1,200㎡', 'suitability_score': 4, 'pros': ['주택가 밀집 지역', '진출입로의 가시 확보가 우수함'], 'cons': ['야간 주차난이 매우 심각함', '주변 진입도로 폭이 6m로 협소하여 대형 차량의 진입이 제한적임']}


1.4 '고속도로 LOS & VMS 추천 시스템' 연계 AI 확장 아이디어

작성해주신 고속도로 혼잡도 추정 프로젝트는 수치 데이터(VDS/하이패스 속도·교통량) 기반의 LOS 산출과 자연어 데이터(VMS 전광판 표지 문구/우회 경로 안내)가 정교하게 융합되는 아키텍처입니다.

이 프로젝트에서 LangChain과 LLM을 더 고도화하여 녹여낼 수 있는 추가 아이디어 3가지입니다

아이디어 A. 
VMS 전광판 규격 맞춤형 문구 자동 생성기 (VMS Text Generator)
개요: 도로공사의 물리적 VMS 전광판 글자 수 제약(예: 한 줄당 8자, 총 2~3줄 내외, 특수문자 제한 등)을 준수하면서, 현재 VDS/하이패스 보정 결과(LOS, 정체 길이, 소요 시간)에 맞는 최적의 표출 문구를 자동 생성합니다.

LLM의 역할: Pydantic 파서를 이용해 제약 조건(글자 수, 줄 수)을 강제한 정형 문자열 생성.

Pydantic 스키마 예시:

In [6]:
class VMSMessage(BaseModel):
    line1: str = Field(description="첫 번째 줄 문구 (8자 이내, 예: [경부선] 양재~판교)")
    line2: str = Field(description="두 번째 줄 문구 (8자 이내, 예: 정체 5km 25분소요)")
    line3: str = Field(description="세 번째 줄 문구 (8자 이내, 예: 국도1호선 우회권장)")

아이디어 B. 센서 이상치(Anomaly) 자동 검증 및 원인 설명 도구
개요: VDS 센서 속도가 갑자기 0km/h로 떨어졌을 때, 이것이 단순 센서 고장/통신 장애인지, 실제 대형 사고 발생으로 인한 전면 차단인지 하이패스 통행시간 데이터 및 기타 텍스트 제보를 융합 판단합니다.

LLM의 역할: 수치 데이터 간의 불일치(VDS는 정체이나 하이패스는 정상 통과 등) 원인을 규명하는 리포트 작성 및 원인 추정(Confidence Score 포함).

아이디어 C. 운전자 맞춤형 자연어 음성/앱 우회 안내 텍스트 생성 (Navi Prompt)
개요: VMS 표지판 문구 외에도, 교통방송(TBN) 원고, 도로공사 앱 알림, 또는 차량 인포테인먼트용 자연어 멘트를 자동으로 생성합니다.

생성 텍스트 예시: "현재 영동고속도로 신갈JCT부터 마성IC 구간 LOS F 등급의 심각한 정체가 발생 중입니다. 하이패스 구간 통행시간 추정 결과 약 40분이 추가 소요되오니, 마성IC 이전 용인대로를 통한 우회도로 이용을 권장합니다."

🚀 [시연 코드 예시] VMS 표지판 문구 자동 생성기 코드
프로젝트 기획안의 3번째 핵심 목표(AI 기반 VMS 문구 및 우회 경로 자동 제너레이션)를 즉시 테스트해 볼 수 있는 파이썬 코드입니다.

In [7]:
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

load_dotenv()

# 1. VMS 규격에 맞춘 Output 스키마 정의
class VMSDisplay(BaseModel):
    line1: str = Field(description="전광판 1행 (최대 8자, 노선 및 정체구간)")
    line2: str = Field(description="전광판 2행 (최대 8자, 정체길이 및 소요시간)")
    line3: str = Field(description="전광판 3행 (최대 8자, 우회도로 권장안)")

# 2. 파서 및 프롬프트 생성
parser = PydanticOutputParser(pydantic_object=VMSDisplay)

prompt = PromptTemplate(
    template="""당신은 한국도로공사 교통관제 AI 시스템입니다.
다음 고속도로 정체 상황 분석 데이터를 바탕으로 VMS(도로전광표지) 표출 문구를 작성하세요.

[규격 조건]
- 각 행은 공백 포함 **최대 8자 이내**여야 합니다.
- 운전자가 80km/h로 주행 중에도 한눈에 알아볼 수 있게 명확하고 직관적이어야 합니다.

[상황 데이터]
- 노선: {route_name}
- 정체 구간: {jam_section}
- 정체 길이: {jam_length}
- LOS 등급: {los_grade}
- 예상 지연시간: {delay_time}
- 추천 우회도로: {detour_route}

{format_instructions}""",
    input_variables=["route_name", "jam_section", "jam_length", "los_grade", "delay_time", "detour_route"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
chain = prompt | llm | parser

# 3. VDS + 하이패스 연계 로직에서 산출된 정체 데이터 입력 예시
traffic_event = {
    "route_name": "경부선 서울방향",
    "jam_section": "수원신갈IC~신갈JCT",
    "jam_length": "6.5km",
    "los_grade": "LOS F (심각)",
    "delay_time": "35분",
    "detour_route": "국도1호선"
}

result = chain.invoke(traffic_event)

print("=== [VMS 전광판 생성 결과] ===")
print(f"[1행]: {result.line1}")
print(f"[2행]: {result.line2}")
print(f"[3행]: {result.line3}")

=== [VMS 전광판 생성 결과] ===
[1행]: 경부선 서울
[2행]: 정체 6.5km
[3행]: 우회: 국도1


실제 공공데이터 API(공공데이터포털, 한국도로공사 등)와 실시간으로 연계해서 시스템을 작동시킬 수 있습니다.



실제 시스템으로 연결하기 위해서는 공공데이터 API 호출 ➔ 데이터 전처리 및 LOS/정체 산출 ➔ LLM 기반 VMS/민원 조치안 자동 생성 흐름의 파이프라인을 구축하면 됩니다.

실제 공공데이터 API 연계 및 처리 흐름
공공데이터포털 API 신청:

[한국도로공사_VDS 차량검지기 실시간 교통량 정보] 및 [하이패스 구간별 통행시간 API] 서비스 신청 후 인증키(Service Key)를 발급받습니다.

requests 라이브러리로 실시간 데이터 수집:

API를 호출해 JSON 또는 XML 형식의 실시간 속도, 교통량, 통행시간 데이터를 받아옵니다.

LLM 체인 전달:

수집된 데이터를 가공하여 앞서 만든 LangChain Pydantic 파서 프롬프트의 입력값으로 전달합니다.

**"도로 위 실시간 센서 및 통행 데이터를 자동으로 분석(LOS)하고, LangChain과 AI를 통해 운전자 안내용 VMS 문구와 관제 조치안을 실시간으로 뚝딱 만들어내는 지능형 스마트 교통 관제 시스템"**입니다.

🎯 프로젝트 주제
VDS·하이패스 실시간 교통 데이터 및 HCM 기준 LOS(도로 서비스 수준)를 연계한 AI 기반 도로전광표지(VMS) 관제 자동화 시스템

💡 시스템의 핵심 목적 (4가지 단계별 목적)
실시간 교통 데이터 수집 및 보완 (fetch_realtime_traffic_data)

한국도로공사 등의 공공데이터 API를 실시간으로 호출하여 고속도로 특정 구간의 속도(Speed), 교통량(Volume), 통행시간(Travel Time) 데이터를 수집합니다.

API 통신 오류나 센서 장애 발생 시 Fallback(더미) 데이터로 자동 전환되도록 예외 처리 구조를 갖추고 있습니다.

객관적 도로 서비스 수준 자동 판정 (calculate_hcm_los)

수집된 실시간 차량 속도(speed)를 HCM(도로용량편람) 표준 기준에 대입하여, 현재 도로가 원활한 상태인지(LOS A~D) 아니면 정체 상태인지(LOS E~F) 객관적인 등급을 자동으로 산출합니다.

LLM 제약 조건 기반 전광판 문구 자동 생성 (PydanticOutputParser + PromptTemplate)

실시간 계산된 정체 데이터(속도, LOS, 지연 시간 등)를 LLM(gpt-4o-mini)에 입력합니다.

Pydantic 스키마를 활용하여 물리적 전광판 규격인 "행당 8자 이내"라는 엄격한 글자 수 제약을 강제하면서, 운전자가 즉각 인지할 수 있는 직관적인 VMS 안내 문구(1~3행)를 자동으로 만들어냅니다.

관제 센터 의사결정 지원 (action_recommendation)

단순 운전자 안내를 넘어, 교통관제 담당자가 현장에서 취해야 할 세부 조치 가이드(예: 갓길차로 개방, 우회도로 유도, 돌발 상황 전파 등)를 함께 도출하여 스마트 교통 관리 업무를 효율화합니다.

In [8]:
import os
import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

load_dotenv()

# ==========================================
# 1. 실제 공공데이터 API 연동 함수 (VDS/하이패스)
# ==========================================
def fetch_realtime_traffic_data(route_no="0010", cz_id="0010CZS010"):
    """
    한국도로공사 VDS/하이패스 실시간 API 호출 예시
    """
    api_key = os.getenv("EX_TRAFFIC_API_KEY") # .env 파일에 저장된 API 키
    url = f"http://data.ex.co.kr/openapi/trafficapi/vdsRealtime"
    
    params = {
        "key": api_key,
        "type": "json",
        "routeNo": route_no,
        "czId": cz_id
    }
    
    try:
        response = requests.get(url, params=params, timeout=5)
        if response.status_code == 200:
            data = response.json()
            # API 응답 구조에 맞춰 실제 데이터 추출 (가상 응답 매핑 처리)
            item = data.get("list", [{}])[0]
            return {
                "route_name": "경부선 서울방향",
                "section_name": item.get("vdsName", "양재~달래내"),
                "speed": float(item.get("speed", 25.4)), # 실시간 평균속도
                "volume": int(item.get("trafficVolume", 1850)), # 교통량
                "travel_time": int(item.get("travelTime", 28)) # 하이패스 보정 통행시간(분)
            }
    except Exception as e:
        print(f"API 호출 실패 또는 더미 데이터 전환: {e}")
        
    # API 연결 실패 시 fallback 더미 데이터 (실시간 상황 가정)
    return {
        "route_name": "경부선 서울방향",
        "section_name": "수원신갈IC~신갈JCT",
        "speed": 22.5,
        "volume": 2100,
        "travel_time": 35
    }

# ==========================================
# 2. LOS(서비스 수준) 자동 판정 로직
# ==========================================
def calculate_hcm_los(speed):
    """HCM 도로용량편람 기준 속도별 LOS 등급 산정"""
    if speed >= 80: return "LOS A"
    elif speed >= 70: return "LOS B"
    elif speed >= 60: return "LOS C"
    elif speed >= 50: return "LOS D"
    elif speed >= 30: return "LOS E (서행)"
    else: return "LOS F (심각)"

# ==========================================
# 3. LLM 기반 VMS 문구 및 우회안 자동 생성
# ==========================================
class VMSRealtimeDisplay(BaseModel):
    line1: str = Field(description="1행: 노선 및 구간 (8자 이내)")
    line2: str = Field(description="2행: 정체 상태 및 지연시간 (8자 이내)")
    line3: str = Field(description="3행: 우회도로 안내 (8자 이내)")
    action_recommendation: str = Field(description="관제 센터 담당자용 세부 조치 가이드")

parser = PydanticOutputParser(pydantic_object=VMSRealtimeDisplay)

prompt = PromptTemplate(
    template="""당신은 고속도로 교통관제 AI 시스템입니다. 
수집된 실시간 교통데이터를 분석하여 VMS 전광판 규격(행당 8자 이내)에 맞는 안내 문구를 작성하고 관제 조치안을 제시하세요.

[실시간 수집 데이터]
- 노선명: {route_name}
- 구간: {section_name}
- 실시간 속도: {speed} km/h
- LOS 등급: {los_grade}
- 측정 통행시간: {travel_time}분

{format_instructions}""",
    input_variables=["route_name", "section_name", "speed", "los_grade", "travel_time"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
chain = prompt | llm | parser

# ==========================================
# 4. 실시간 모니터링 실행 파이프라인
# ==========================================
# ① 실시간 API 데이터 수집
raw_data = fetch_realtime_traffic_data()

# ② HCM 기준 LOS 자동 계산
los_result = calculate_hcm_los(raw_data["speed"])
raw_data["los_grade"] = los_result

# ③ LLM 파이프라인 실행
parsed_result = chain.invoke(raw_data)

# ④ 결과 출력
print("=== [실시간 데이터 연계 VMS 관제 시스템] ===")
print(f"📊 실시간 수집 속도: {raw_data['speed']} km/h ({los_result})")
print("-" * 40)
print(f"📺 [VMS 1행]: {parsed_result.line1}")
print(f"📺 [VMS 2행]: {parsed_result.line2}")
print(f"📺 [VMS 3행]: {parsed_result.line3}")
print("-" * 40)
print(f"💡 관제 조치안: {parsed_result.action_recommendation}")

=== [실시간 데이터 연계 VMS 관제 시스템] ===
📊 실시간 수집 속도: 22.5 km/h (LOS F (심각))
----------------------------------------
📺 [VMS 1행]: 경부선 서울
📺 [VMS 2행]: 정체 심각 35분
📺 [VMS 3행]: 우회: 1번 도로
----------------------------------------
💡 관제 조치안: 신갈JCT 부근 교통량 모니터링 및 우회 안내 강화


1.5 C-ITS 스마트 교통 관제 및 AI 대응 시스템

이 통합 시스템의 실무적 가치
데이터 전처리 파이프라인 (preprocess_vds_data): 센서 오작동이나 통신 두절 상황을 필터링(status_flag)하고, HCM 기준을 자동 계산하여 오류 없는 신뢰성 높은 데이터를 AI에 공급합니다.

민원 자동 분류 알고리즘 (ComplaintAnalysis): 들어오는 수많은 민원 텍스트를 구조화하여 담당자가 긴급도 순으로 신속하게 처리할 수 있게 돕습니다.

C-ITS 지능형 관제 에이전트 (IncidentResponseReport): VDS 수치 데이터와 CCTV 정성적 관제 로그를 융합하여, 사고 발생 시 운전자 안내 전광판(VMS) 문구부터 현장 출동반의 구체적인 조치 가이드까지 원스톱으로 산출합니다.

1. 프로젝트 주제 (Project Title / Topic)
"멀티모달 센서(VDS·하이패스·CCTV) 데이터 융합과 LangChain 기반 LLM 에이전트를 활용한 실시간 C-ITS 도로 혼잡도(LOS) 예측 및 지능형 돌발 대응 자동화 시스템"

2. 프로젝트 목적 (Project Objectives)
센서 데이터 신뢰도 극대화: 기존 단일 VDS(차량검지기) 센서의 오작동, 통신 장애, 속도 왜곡 문제를 하이패스 실제 통행시간 데이터와 상호 교차 검증하여 정체 판단의 정확도 향상

C-ITS 기반 객관적 교통 서비스 수준(LOS) 자동 산정: 도로용량편람(HCM) 표준 기준에 따라 실시간 노선별 서비스 수준(LOS A~F)을 자동 판정하여 선제적 교통 관제 기반 마련

돌발 상황 대응 자동화 및 신속성 확보: CCTV 관제 로그와 실시간 수치 데이터를 AI 에이전트에 연계하여, 사고 및 정체 발생 시 운전자 대상 VMS(전광판) 안내 문구와 현장 대응 가이드를 즉각 제너레이션

행정 업무 효율화: 비구조화된 시민 교통 민원 텍스트를 자동으로 분류·구조화하여 지자체 및 관제 센터의 우선순위 조치 의사결정 지원

3. 프로젝트 개선 방안 (Improvements & Scalability)
멀티모달 융합 고도화: 단순 텍스트/숫자 결합을 넘어, CCTV 이미지 프레임이나 영상 객체 인식(YOLO 등) 결과를 LLM 비전 모델(Vision LLM)과 연계하여 사고 심각도를 시각적으로 정밀 판독

실시간 스트리밍 아키텍처 도입: 단발성 API 호출(requests)을 넘어 Kafka나 FastAPI WebSockets 기반의 실시간 데이터 파이프라인으로 확장하여 지연 없는(Real-time) 관제 대시보드 구축

자동 우회 경로 알고리즘 결합: OSRM 또는 Naver/Kakao Maps Directions API를 연계하여, 정체 발생 시 주변 국도 및 우회 도로의 실시간 소요 시간을 비교·안내하는 기능 추가

4. 프로젝트의 한계점 및 고려사항 (Limitations & Challenges)
공공데이터 API 응답 지연 및 레이트 리밋(Rate Limit): 공공데이터포털 및 도로공사 API의 호출 횟수 제한(Quota)이나 서버 응답 지연 문제로 인해 초 단위의 엄격한 실시간 제어에는 물리적 제약 존재

LLM 환각(Hallucination) 현상 및 비용: 프롬프트 제약(Pydantic)으로 제어하고 있으나, 긴급 상황에서 AI가 규격에 어긋나는 VMS 문구나 비현실적인 조치 가이드를 생성할 확률이 완전 제로(0)는 아니며, 대규모 실시간 호출 시 API 비용 발생

현장 인프라와의 연동 보안 문제: 실제 지자체(예: 군산시 도시통합센터 등)의 폐쇄망 교통 신호 제어 시스템(SCATS/UTIS) 및 VMS 하드웨어 장비와 직접 연동하려면 엄격한 보안 인증 및 표준 프로토콜 준수 필요

In [9]:
from typing import List, Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

load_dotenv()

# ==========================================
# [파트 1] C-ITS 실시간 VDS 데이터 전처리 및 LOS 예측 모델
# ==========================================
def preprocess_vds_data(raw_sensor_data: dict) -> dict:
    """
    실시간 VDS 검지기 센서 데이터를 전처리하고 HCM 기준 LOS 산출 및 이상치 보정
    """
    speed = float(raw_sensor_data.get("speed", 0.0))
    volume = int(raw_sensor_data.get("volume", 0))
    occupancy = float(raw_sensor_data.get("occupancy", 0.0))
    
    # 1. 이상치(센서 통신 오류 등 0km/h 및 비정상 수치) 보정 로직
    if speed <= 0 or occupancy > 80:
        status_flag = "SENSOR_ANOMALY_OR_GRIDLOCK"
    else:
        status_flag = "NORMAL_OPERATION"

    # 2. HCM 도로용량편람 기준 서비스 수준(LOS) 산정
    if speed >= 80: los = "LOS A"
    elif speed >= 70: los = "LOS B"
    elif speed >= 60: los = "LOS C"
    elif speed >= 50: los = "LOS D"
    elif speed >= 30: los = "LOS E (서행)"
    else: los = "LOS F (정체/위험)"

    return {
        "section_id": raw_sensor_data.get("section_id", "SEC-01"),
        "speed": speed,
        "volume": volume,
        "occupancy": occupancy,
        "status_flag": status_flag,
        "los_grade": los
    }


# ==========================================
# [파트 2] 현장 교통 민원 자동 분류 및 우선순위 알고리즘
# ==========================================
class ComplaintAnalysis(BaseModel):
    category: str = Field(description="민원 분류 (예: 신호체계, 보행자안전, 불법주정차, 시설물파손)")
    urgency_level: int = Field(description="긴급도 (1~5점 중 정수)")
    action_plan: List[str] = Field(description="현장 출동 및 조치 검토 사항 리스트")

complaint_parser = PydanticOutputParser(pydantic_object=ComplaintAnalysis)
complaint_prompt = PromptTemplate(
    template="다음 현장 교통 민원을 분석하여 분류하고 조치안을 도출하세요.\n민원내용: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": complaint_parser.get_format_instructions()}
)


# ==========================================
# [파트 3] VDS/CCTV 연계 AI 에이전트 기반 돌발 상황 대응 리포트 생성
# ==========================================
class IncidentResponseReport(BaseModel):
    incident_type: str = Field(description="돌발 상황 유형 (예: 차량 추돌 사고, 낙하물, 극심한 정체)")
    severity: str = Field(description="심각도 수준 (Low, Medium, Critical)")
    vms_message: List[str] = Field(description="전광판 표출용 3행 이내 문구 리스트")
    response_guide: List[str] = Field(description="관제 센터 및 현장 출동반 대응 가이드")
    estimated_recovery_min: int = Field(description="예상 정상화 소요 시간 (분 단위 숫자)")

incident_parser = PydanticOutputParser(pydantic_object=IncidentResponseReport)
incident_prompt = PromptTemplate(
    template="""당신은 C-ITS 스마트 교통 관제 AI 마스터 에이전트입니다.
아래의 실시간 VDS 센서 분석 결과와 CCTV 관제 로그를 바탕으로 종합 대응 리포트와 VMS 안내 문구를 생성하세요.

[VDS 센서 데이터 분석 결과]
- 구간 ID: {section_id}
- 실시간 속도: {speed} km/h
- 교통량: {volume} 대/h
- 서비스 수준(LOS): {los_grade}
- 센서 상태 플래그: {status_flag}

[CCTV 현장 관제 로그]
- {cctv_log}

{format_instructions}""",
    input_variables=["section_id", "speed", "volume", "los_grade", "status_flag", "cctv_log"],
    partial_variables={"format_instructions": incident_parser.get_format_instructions()}
)


# ==========================================
# [통합 실행 파이프라인]
# ==========================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# 1. 시뮬레이션 데이터 수입 (실제 환경에서는 API에서 실시간 수집)
raw_vds_input = {
    "section_id": "경부선_서울방향_104K",
    "speed": 18.5,
    "volume": 2450,
    "occupancy": 75.2
}
cctv_observation_log = "2차로 상에 승용차 단독 추돌 사고 발생 후 후속 차량들이 급감속하며 정체 꼬리가 빠르게 길어지고 있음. 갓길 차로 유도 필요."
citizen_complaint_text = "사고 지점 후방 1km 구간에서 차량들이 엉켜서 옴짝달싹 못 하고 있습니다. 빠른 통제와 안내가 필요해요."

print("=== [1단계] VDS 실시간 센서 데이터 전처리 및 LOS 판정 ===" )
processed_vds = preprocess_vds_data(raw_vds_input)
print(processed_vds)

print("\n=== [2단계] 현장 교통 민원 자동 분류 결과 ===")
complaint_chain = complaint_prompt | llm | complaint_parser
complaint_result = complaint_chain.invoke({"text": citizen_complaint_text})
print(complaint_result.model_dump())

print("\n=== [3단계] C-ITS VDS/CCTV 연계 AI 돌발 대응 리포트 및 VMS 생성 ===")
incident_chain = incident_prompt | llm | incident_parser
payload = {**processed_vds, "cctv_log": cctv_observation_log}
incident_result = incident_chain.invoke(payload)

print(incident_result.model_dump())

=== [1단계] VDS 실시간 센서 데이터 전처리 및 LOS 판정 ===
{'section_id': '경부선_서울방향_104K', 'speed': 18.5, 'volume': 2450, 'occupancy': 75.2, 'status_flag': 'NORMAL_OPERATION', 'los_grade': 'LOS F (정체/위험)'}

=== [2단계] 현장 교통 민원 자동 분류 결과 ===
{'category': '사고', 'urgency_level': 5, 'action_plan': ['사고 지점 후방 1km 구간에 대한 교통 통제 실시', '차량 운전자에게 상황 안내 및 우회 도로 안내', '사고 처리 및 차량 견인 지원 요청', '현장 경찰 및 교통 관리 인력 배치', '교통 상황 모니터링 및 추가 조치 필요 시 대응']}

=== [3단계] C-ITS VDS/CCTV 연계 AI 돌발 대응 리포트 및 VMS 생성 ===
{'incident_type': '차량 추돌 사고', 'severity': 'Critical', 'vms_message': ['경부선 서울방향 104K', '2차로 사고 발생', '갓길로 우회하세요'], 'response_guide': ['사고 현장에 즉시 출동하여 안전 조치 실시', '후속 차량의 갓길 유도 및 정체 해소 작업 진행', '교통량 모니터링 및 추가 지원 요청'], 'estimated_recovery_min': 30}
